# 17.4 Linting and Formatting with `ruff`

**Prerequisites:** 17.2 pyproject.toml, 1.2 Python Basic (PEP 8), 15 Testing and Debugging  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- The difference between a **formatter** and a **linter** — and why you want both
- `ruff check` and `ruff format`, on genuinely messy code
- Rule codes, rule families, and `ruff rule` to explain any of them
- 🔴 Autofix — and the difference between safe and **unsafe** fixes
- Choosing rules: `select`, `extend-select`, `ignore`, `per-file-ignores`
- 🔴 `# noqa` written three ways, two of them wrong
- `pre-commit`, and where linting belongs in CI (**15.6**)
- What a linter catches that a type checker and tests do not

---

## Two different jobs

| | **Formatter** | **Linter** |
|---|---|---|
| Question | *how does this look?* | *is this a good idea?* |
| Changes | whitespace, quotes, line breaks | flags problems; sometimes fixes them |
| Opinionated | totally — no arguments | configurable |
| Example | `f( x,y )` → `f(x, y)` | "you never use `os`" |
| Tool | `ruff format` (or `black`) | `ruff check` (or `flake8`, `pylint`) |

The formatter ends style arguments by removing the choice. The linter finds real problems —
unused imports, mutable default arguments (**4.1**), bare `except` (**6.1**), comparisons to
`None` with `==`.

**`ruff` is both**, written in Rust, and fast enough that the whole thing runs on save. It
replaced a stack of tools that used to be separate:

| Old tool | Now |
|---|---|
| `flake8` + plugins | `ruff check` |
| `black` | `ruff format` |
| `isort` | `ruff check --select I` |
| `pyupgrade` | `--select UP` |
| `bandit` | `--select S` |

🔴 One tool, one config table (**17.2**), one pass over the files.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py174_"))


def write(rel, source, root=None):
    path = (root or WORK) / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return path


def ruff(*args, cwd=None, quiet=False):
    """Run ruff and return its output, with the command echoed."""
    done = subprocess.run([sys.executable, "-m", "ruff", *args],
                          cwd=cwd or WORK, capture_output=True, text=True,
                          encoding="utf-8", errors="replace", timeout=300)
    body = (done.stdout + done.stderr).strip() or "(nothing to report)"
    if quiet:
        return body
    return (f"$ ruff {' '.join(args)}\n" + "-" * 68 + "\n" + body
            + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


print("scratch:", WORK)
print("ruff   :", subprocess.run([sys.executable, "-m", "ruff", "--version"],
                                 capture_output=True, text=True).stdout.strip())

## A file with real problems

Every issue below is one you will meet in someone's codebase — including your own.

In [ ]:
MESSY = r"""
    import os, sys
    import json
    from collections import OrderedDict


    def process_rows( rows,seen = [] ):
        result = {}
        for i in range(len(rows)):
            row = rows[i]
            if row == None:
                continue
            if row not in seen:
                seen.append(row)
            result[row] = len( row )
        try:
            payload = json.loads("{}")
        except:
            pass
        return result
"""

write("messy.py", MESSY)
print(ruff("check", "messy.py", "--output-format=concise"))

Read that list against the code — every one is a real defect, not a style
preference:

| Code | Problem | Why it matters |
|---|---|---|
| `I001` | imports unsorted | churn in every diff that touches imports |
| `F401` | `os`, `sys`, `OrderedDict` unused | dead weight, and a hint the code moved on |
| `B006` | 🔴 **mutable default `seen = []`** | the classic shared-state bug from **4.1** |
| `F841` | `payload` assigned, never used | usually a half-finished edit |
| `E722` | bare `except:` | swallows `KeyboardInterrupt` — **6.1** |
| `S110` | `try`/`except`/`pass` | an error silently discarded (**15.7**) |

`--output-format=concise` gives one line per problem. The default format prints the source with
carets and a suggested fix, which is better in a terminal and too wide for a notebook.

> 🔴 **`B006` alone justifies a linter.** `def f(seen=[])` creates the list **once, at
> definition time**, and every call shares it. **4.1** demonstrates the bug; ruff finds it in
> code you have not read.

## Explaining a rule

You will meet codes you do not recognise. `ruff rule` prints the reasoning, with examples.

In [ ]:
explanation = ruff("rule", "B006", quiet=True)
print(explanation[:1500])

That is the whole argument for the rule, including the fix. No web search.

## Autofix, and 🔴 the safe/unsafe distinction

`ruff check --fix` applies fixes. `--diff` shows what it *would* do without touching anything —
always look first.

Ruff divides fixes into two kinds, and the distinction is important:

| | **Safe** | **Unsafe** |
|---|---|---|
| Changes behaviour | never | 🔴 **possibly** |
| Applied by | `--fix` | `--fix --unsafe-fixes` |
| Example | sorting imports | deleting an unused variable that had a side effect |

🔴 Removing `payload = json.loads("{}")` looks safe — the variable is unused. But
`json.loads` could raise, and deleting the call removes that. Ruff classes it unsafe and
leaves it alone unless you ask.

In [ ]:
print(ruff("check", "messy.py", "--diff"))
print()
print("what --fix would leave behind:")
print(ruff("check", "messy.py", "--statistics"))

`Would fix 3 errors (2 additional fixes available with --unsafe-fixes)`.

The statistics view is how you triage a large codebase: fix the biggest category first, since
it is usually one mechanical change repeated everywhere — the same tactic **16.6** used for type
errors.

## The formatter

`ruff format` is a **`black`-compatible** formatter. It has essentially no options, which is
the point: nobody argues about it.

In [ ]:
print(ruff("format", "--diff", "messy.py"))

Spacing normalised, and nothing else touched. The formatter does **not** fix
the unused imports or the mutable default — those are the linter's job, and this is exactly why
you run both.

🔴 **Order matters in CI:** format first, then lint. A formatter can introduce a line-length
issue; a linter cannot introduce a formatting one.

```bash
ruff format .          # rewrite
ruff check --fix .     # then lint
```

## Choosing rules

Ruff implements **over 800** rules from dozens of linters. By default it enables a small, safe
subset (`E4`, `E7`, `E9`, `F`). You opt in to more.

| Family | Source | Catches |
|---|---|---|
| `E`, `W` | pycodestyle | PEP 8 style (**1.2**) |
| `F` | pyflakes | unused imports and names, real errors |
| `I` | isort | import ordering |
| `B` | flake8-bugbear | 🔴 likely bugs — mutable defaults, loop-variable capture |
| `UP` | pyupgrade | modernise for your `target-version` |
| `SIM` | flake8-simplify | needlessly complex constructs |
| `S` | bandit | security patterns |
| `ANN` | flake8-annotations | missing type hints (**16**) |
| `D` | pydocstyle | docstring conventions |
| `RUF` | ruff's own | including `RUF100`, unused `noqa` |

```toml
[tool.ruff.lint]
select = ["E", "F", "W", "I", "B", "UP", "SIM"]
ignore = ["E501"]                       # line length - the formatter handles it

[tool.ruff.lint.per-file-ignores]
"tests/*" = ["S101"]                    # assert is the point of a test (15.1)
"__init__.py" = ["F401"]                # re-exports are deliberate
```

🔴 **Those two per-file ignores are near-universal.** `assert` in tests is flagged by bandit's
`S101`; re-exports in `__init__.py` look like unused imports. Without them you fight your own
linter.

In [ ]:
write("pyproject.toml", r"""
    [tool.ruff]
    line-length = 100
    target-version = "py312"

    [tool.ruff.lint]
    select = ["E", "F", "W", "I", "B", "UP", "SIM"]

    [tool.ruff.lint.per-file-ignores]
    "tests/*" = ["S101"]
    "__init__.py" = ["F401"]
""")

write("pkg/__init__.py", r"""
    from pkg.retry import retry_delay
""")
write("pkg/retry.py", r"""
    from typing import Dict, List, Optional


    def retry_delay(attempt: int, ceiling: Optional[float] = None) -> float:
        cap = ceiling if ceiling is not None else 30.0
        delay = 1.0
        for _ in range(attempt):
            delay = delay * 2
        return min(delay, cap)


    def schedule(attempts: int) -> List[float]:
        out: List[float] = []
        for n in range(attempts):
            out.append(retry_delay(n))
        return out


    def index(rows: List[str]) -> Dict[str, int]:
        result = {}
        for row in rows:
            if row in result.keys():
                continue
            result[row] = len(row)
        return result
""")
write("tests/test_retry.py", r"""
    from pkg.retry import retry_delay


    def test_ceiling():
        assert retry_delay(9) == 30.0
""")

print(ruff("check", ".", "--output-format=concise"))

Look at what the wider rule set found:

- **`UP035` / `UP006` / `UP045`** — `Dict`, `List` and `Optional` from `typing` are the old
  spellings. Since 3.9 and 3.10 the builtins and `|` do the job (**16.2**). `target-version =
  "py312"` is what lets ruff know this is safe to say.
- **`SIM118`** — `if row in result.keys()` should be `if row in result` (**2.5**).
- **No `F401` for `pkg/__init__.py`** — the per-file ignore did its job; the re-export is
  not reported as an unused import. (The `S101` entry is inert here because `S` is not in
  `select` — it only starts mattering once you enable the security family.)

🔴 The `UP` family is the one that keeps a codebase from ageing. It is how you find every
`Optional[X]`, `%`-format string and `os.path` call still lying around after an upgrade.

## Fixing it

In [ ]:
print(ruff("check", ".", "--fix", "--output-format=concise"))
print()
print("pkg/retry.py after --fix:")
print(textwrap.indent((WORK / "pkg" / "retry.py").read_text(encoding="utf-8"), "   "))

The imports are gone and the annotations are modern — `float | None`,
`list[float]`, `dict[str, int]` — because `target-version` said 3.12 was the floor.

## 🔴 `# noqa`, three ways

Sometimes a rule is wrong for one line. `# noqa` silences it — and how you write it matters
enormously.

In [ ]:
write("noqa_demo.py", r"""
    import os  # noqa
    import sys  # noqa: F401
    import json  # noqa: E501
""")

print(ruff("check", "noqa_demo.py", "--select", "F,E,RUF100",
           "--output-format=concise"))

Three imports, three outcomes:

| Line | Written | Result |
|---|---|---|
| `import os  # noqa` | bare | 🔴 **silences everything on that line, forever** — no error reported |
| `import sys  # noqa: F401` | targeted | silences exactly that rule; correct |
| `import json  # noqa: E501` | wrong code | F401 **still reported**, plus `RUF100` for the useless directive |

🔴 **A bare `# noqa` is the linting equivalent of a bare `except:` (**6.1**) or a bare
`# type: ignore` (**16.1**).** It hides the problem you knew about *and* every problem that
appears on that line later.

**`RUF100` is the guard rail** — it flags `noqa` directives that no longer suppress anything, so
they cannot outlive their reason. It is the direct counterpart of mypy's
`--warn-unused-ignores` (**16.1**). Turn it on:

```toml
[tool.ruff.lint]
extend-select = ["RUF100"]
```

## Where linting belongs

**Three places**, each with a different job:

| Where | How | Why |
|---|---|---|
| **Your editor** | ruff LSP / extension | instant; you fix it as you type |
| **A pre-commit hook** | `pre-commit` | 🔴 catches it before it reaches a branch |
| **CI** | `ruff check .` | the one that is *binding* (**15.6**) |

```yaml
# .pre-commit-config.yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.5.0
    hooks:
      - id: ruff
        args: [--fix]
      - id: ruff-format
```

```bash
pip install pre-commit && pre-commit install
```

🔴 **Pin the ruff version** — in `pyproject.toml`, in the hook, and in CI. A new release adds
rules, which is good, but it should be a deliberate upgrade rather than a red build on somebody
else's pull request. Exactly the argument made for mypy in **16.6**.

## What a linter catches that other tools do not

The three checkers overlap far less than people assume:

| Problem | Linter | Type checker (**16**) | Tests (**15**) |
|---|---|---|---|
| Unused import | ✅ | ❌ | ❌ |
| Mutable default argument | ✅ | ❌ | only if you hit it |
| Bare `except:` | ✅ | ❌ | ❌ |
| `if x == None` | ✅ | ❌ | ❌ |
| Wrong argument type | ❌ | ✅ | only that path |
| `None` where a value was expected | ❌ | ✅ | only that path |
| **Wrong answer** | ❌ | ❌ | ✅ |

🔴 **Nothing in the first two columns can tell you the code does the right thing.** A linter
finds *suspicious shapes*, a type checker finds *inconsistent types*, and only a test knows what
the answer should be. Run all three (**15.6**).

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **A bare `# noqa`.** It silences every rule on that line, including ones that appear later. Always `# noqa: RULE`.
2. **Never enabling `RUF100`.** Stale `noqa` directives then accumulate and hide real findings — the same failure as an unused `type: ignore` (**16.1**).
3. **Running the linter before the formatter.** Format first; a formatter can create a lint finding, not the reverse.
4. 🔴 **Applying `--unsafe-fixes` without reading the diff.** Unsafe means it can change behaviour — deleting a call that had a side effect, for instance.
5. **Leaving `select` at the default.** You get a small subset and miss `B`, `I`, `UP` and `SIM`, which are where most of the value is.
6. **Enabling every rule at once on an existing codebase.** Thousands of findings, and the team turns it off — the same mistake as `--strict` in **16.6**.
7. **Forgetting `per-file-ignores` for tests and `__init__.py`.** You then fight your own linter over `assert` and re-exports.
8. **Not pinning the linter version.** A new release adds rules and reddens an unrelated PR.
9. **Treating a clean lint as correctness.** It says nothing about whether the answer is right.

## Best Practices

- Use `ruff` for both jobs: `ruff format` then `ruff check --fix`.
- Configure it in `pyproject.toml` (**17.2**) so editor, hook and CI agree.
- Start with `select = ["E", "F", "W", "I", "B", "UP", "SIM"]` and widen when that is clean.
- Add the two near-universal `per-file-ignores`: `S101` in tests, `F401` in `__init__.py`.
- Turn on `RUF100` so silenced rules cannot outlive their reason.
- Read `--diff` before `--fix`, and never use `--unsafe-fixes` unattended.
- Set `target-version` honestly — it is what makes the `UP` family safe and useful.
- Run it in your editor, in a pre-commit hook, and in CI: fast, early, binding.
- Pin the version, and upgrade deliberately.

## Practice Exercises

Try these before moving on.

1. Run `ruff check` on a project of your own with the default rules, then with `--select E,F,W,I,B,UP,SIM`. How many more findings, and how many are real?
2. 🔴 Find a `B006` mutable-default finding in real code (or write one) and demonstrate the bug it predicts, as **4.1** does.
3. Use `ruff rule` on three codes you do not recognise. Which one surprised you?
4. Run `--diff`, then `--fix`, then `--fix --unsafe-fixes` on the same messy file. What changed at each step, and would you have committed all of it?
5. Add `per-file-ignores` for your tests directory and confirm `S101` stops firing there but still fires elsewhere.
6. 🔴 Put a bare `# noqa` on a line, then introduce a *different* error on that same line. Confirm the linter says nothing. Now switch to `# noqa: F401` and watch it reappear.
7. Turn on `RUF100` and count the stale `noqa` comments in a codebase you did not write.
8. Set up a pre-commit hook with ruff, then try to commit badly formatted code.
9. **Interview question:** you have a linter, a type checker and a test suite, all green. What classes of bug could still be present?

---

## Version notes

| Version | Change |
|---|---|
| **ruff 0.16** | the version used here; the formatter is stable and `black`-compatible |
| **ruff 0.5+** | `[tool.ruff.lint]` is the settings table; older configs put rules directly under `[tool.ruff]` and now warn |
| **Python 3.12** | `target-version = "py312"` lets the `UP` family suggest `list[str]`, `X | Y` and PEP 695 generics (**16.3**) |
| **Python 3.9 / 3.10** | the changes `UP006` and `UP007` are about — builtin generics and `X | Y` unions |

> **The tools ruff replaced** — `flake8`, `black`, `isort`, `pyupgrade`, `bandit`, `pydocstyle` —
> all still work, and plenty of projects use them. Ruff's advantage is being one binary, one
> config and roughly two orders of magnitude faster; the *rules* are the same rules.

## Where next

| Notebook | Covers |
|---|---|
| **17.5** | profiling and performance — the last piece of the toolchain |

## Related

- **17.2 pyproject.toml** — the `[tool.ruff]` table configured here
- **1.2 Python Basic** — PEP 8, which the `E`/`W` families enforce
- **4.1 Functions** — the mutable default argument that `B006` predicts
- **6.1 Exception Handling** — bare `except`, which `E722` flags
- **16.1 / 16.6** — `type: ignore` and `--warn-unused-ignores`, the exact counterparts of
  `# noqa` and `RUF100`
- **15.6 Testing in Practice** — the CI pipeline all three tools belong in